# Module 0.3: Bash Scripting for Bioinformatics

---

### Learning Objectives

By the end of this module, you will be able to:
- Write and execute Bash scripts to automate repetitive tasks
- Use variables, conditionals, loops, and functions
- Process multiple files in batch (FASTQ, BAM, VCF)
- Build robust scripts with error handling and input validation
- Apply text processing tools (awk, sed, cut) inside scripts
- Create reusable analysis pipeline scripts

**Prerequisites:** Linux command line (Module 0.1)

**Estimated time:** 3-4 hours

---

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook teaches core computational literacy used before any serious bioinformatics analysis: reproducible shell workflows, stats reasoning, and environment hygiene.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Difference between command syntax and conceptual model (what changes state vs what only inspects).
- Interpreting statistical output: p-value alone is not enough without effect size and assumptions.
- Reproducibility: the same command can yield different outputs if input files or working directory differ.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: Module 0.2: Git Version Control for Scientists](../../Tier_0_Computational_Foundations/02_Git_Version_Control/01_git_version_control.ipynb) | [Next: Character Encodings and Binary Data in Bioinformatics →](../../Tier_0_Computational_Foundations/04_File_Encodings/01_character_encodings.ipynb)

---

## Why Bash Scripting for Bioinformatics?

A typical sequencing experiment generates data for 50-200 samples. Each sample needs
to go through the same pipeline: quality control, trimming, alignment, quantification.
Doing this manually is not feasible.

```
Manual approach (100 samples):

  fastqc sample_001.fastq.gz
  fastqc sample_002.fastq.gz
  fastqc sample_003.fastq.gz
  ... (97 more times, hoping you do not make a typo)

Scripted approach:

  for sample in data/*.fastq.gz; do
      fastqc "$sample" -o results/qc/
  done

  3 lines. Zero typos. Runs overnight. Fully reproducible.
```

Bash scripts also serve as documentation -- they record *exactly* what commands
were run, with what parameters, in what order.

---

## 1. Your First Bash Script

A Bash script is just a text file containing commands that the shell executes in order.

In [ ]:
%%bash
# Create a simple script
cat > /tmp/hello_bioinfo.sh << 'EOF'
#!/bin/bash
# My first bioinformatics script
#
# The first line (#!/bin/bash) is called the "shebang".
# It tells the system which interpreter to use.

echo "Welcome to Bioinformatics!"
echo "Today is $(date '+%Y-%m-%d %H:%M')"
echo "User: $USER"
echo "Working directory: $(pwd)"
echo "System: $(uname -s)"
EOF

# Make it executable
chmod +x /tmp/hello_bioinfo.sh

# Run it
/tmp/hello_bioinfo.sh

### Script structure conventions

```bash
#!/bin/bash
# ============================================================
# Script: pipeline_step1.sh
# Description: Run FastQC on all raw FASTQ files
# Author: Your Name
# Date: 2024-01-15
# Usage: ./pipeline_step1.sh <input_dir> <output_dir>
# ============================================================

set -euo pipefail    # Strict error handling (explained later)

# --- Configuration ---
THREADS=8
MIN_QUALITY=20

# --- Functions ---
log() { echo "[$(date '+%H:%M:%S')] $1"; }

# --- Main logic ---
log "Starting analysis..."
# ... your commands ...
log "Done!"
```

Two ways to run a script:
```bash
chmod +x script.sh    # Make executable (once)
./script.sh           # Run directly

bash script.sh        # Run with bash (does not need chmod)
```

---

## 2. Variables

Variables store values that you reference throughout your script. This avoids
hard-coding paths and parameters, making scripts reusable.

In [ ]:
%%bash
# === Variable assignment ===
# CRITICAL: NO SPACES around the = sign!
# RIGHT:  name="value"
# WRONG:  name = "value"   (bash interprets this as running a command called 'name')

sample_name="BRCA_patient_001"
num_threads=8
genome_file="/references/GRCh38.fa"
output_dir="results/aligned"

# === Using variables ===
echo "Processing: $sample_name"
echo "Threads: ${num_threads}"           # ${} is safer when appending text
echo "Output: ${output_dir}/${sample_name}.bam"

# === Command substitution -- capture command output in a variable ===
current_date=$(date +%Y-%m-%d)
num_cpus=$(nproc)
echo "Date: $current_date"
echo "Available CPUs: $num_cpus"

In [ ]:
%%bash
# === Special variables ===
# These are set automatically by Bash

cat > /tmp/show_args.sh << 'EOF'
#!/bin/bash
echo "Script name:         $0"
echo "First argument:      $1"
echo "Second argument:     $2"
echo "All arguments:       $@"
echo "Number of arguments: $#"
echo "Exit code of last command: $?"
echo "Process ID:          $$"
EOF
chmod +x /tmp/show_args.sh

/tmp/show_args.sh sample_001.fastq.gz /data/output

### Quoting rules (this trips up everyone)

```bash
name="World"

echo "Hello $name"     # Double quotes: variables ARE expanded  -> Hello World
echo 'Hello $name'     # Single quotes: variables NOT expanded  -> Hello $name
echo Hello $name       # No quotes: works, but breaks on spaces in values!
```

**Golden rule: Always double-quote your variables.** `"$variable"` not `$variable`.

Why? If `file="my data.bam"`, then `ls $file` becomes `ls my data.bam` (two arguments),
while `ls "$file"` correctly becomes `ls "my data.bam"` (one argument).

### Bioinformatics example: Configurable pipeline

Using variables to make a pipeline configurable is the first step toward
reproducibility. Anyone can see and change the parameters at the top of the script.

In [ ]:
%%bash
cat > /tmp/pipeline_config.sh << 'EOF'
#!/bin/bash
# === RNA-Seq Pipeline Configuration ===

# Directories
PROJECT_DIR="/home/user/rna_seq_project"
RAW_DATA="${PROJECT_DIR}/00_raw_data"
QC_DIR="${PROJECT_DIR}/01_qc"
TRIMMED="${PROJECT_DIR}/02_trimmed"
ALIGNED="${PROJECT_DIR}/03_aligned"
COUNTS="${PROJECT_DIR}/04_counts"

# Reference files
GENOME="/references/GRCh38.fa"
ANNOTATION="/references/gencode.v38.gtf"
STAR_INDEX="/references/STAR_index"

# Parameters
THREADS=16
MIN_READ_LENGTH=35
QUALITY_THRESHOLD=20
ADAPTER="AGATCGGAAGAGC"

echo "Pipeline configuration:"
echo "  Project:     ${PROJECT_DIR}"
echo "  Genome:      ${GENOME}"
echo "  Threads:     ${THREADS}"
echo "  Min length:  ${MIN_READ_LENGTH}"
echo "  Min quality: ${QUALITY_THRESHOLD}"
EOF
bash /tmp/pipeline_config.sh

---

## 3. Conditionals (if/elif/else)

Conditionals let your script make decisions: does this file exist? Did the last
command succeed? Are there enough arguments?

In [ ]:
%%bash
# Basic syntax
number=42

if [[ $number -gt 100 ]]; then
    echo "Greater than 100"
elif [[ $number -gt 10 ]]; then
    echo "Between 11 and 100"
else
    echo "10 or less"
fi

### Comparison operators

```
NUMERIC                       STRING
  -eq   equals                  ==    equals
  -ne   not equals              !=    not equals
  -lt   less than               -z    string is empty
  -le   less or equal           -n    string is not empty
  -gt   greater than
  -ge   greater or equal

FILE TESTS                    LOGICAL
  -e file    exists             &&    AND
  -f file    is regular file    ||    OR
  -d file    is directory       !     NOT
  -s file    size > 0
  -r file    is readable
  -w file    is writable
  -x file    is executable
```

**Important:** Use `[[ ]]` (double brackets) in Bash, not `[ ]`. Double brackets
are safer with spaces and special characters.

In [ ]:
%%bash
# File existence checks -- you will use these constantly

test_file="/tmp/hello_bioinfo.sh"

if [[ -f "$test_file" ]]; then
    echo "File exists: $test_file"
else
    echo "File NOT found: $test_file"
fi

# Check if a directory exists, create it if not
output_dir="/tmp/test_results"
if [[ ! -d "$output_dir" ]]; then
    echo "Creating output directory: $output_dir"
    mkdir -p "$output_dir"
fi

# Check if a command/tool is available
if command -v python3 &> /dev/null; then
    echo "Python3 is installed: $(python3 --version)"
else
    echo "Python3 is NOT installed"
fi

### Bioinformatics example: Input validation script

In [ ]:
%%bash
cat > /tmp/validate_input.sh << 'SCRIPT'
#!/bin/bash
# Validate input files before running a pipeline

input_file=$1

# Check if argument was provided
if [[ -z "$input_file" ]]; then
    echo "ERROR: No input file specified"
    echo "Usage: $0 <fastq_file>"
    exit 1
fi

# Check if file exists
if [[ ! -f "$input_file" ]]; then
    echo "ERROR: File not found: $input_file"
    exit 1
fi

# Check if file is not empty
if [[ ! -s "$input_file" ]]; then
    echo "ERROR: File is empty: $input_file"
    exit 1
fi

# Check file extension
if [[ "$input_file" != *.fastq && "$input_file" != *.fastq.gz && "$input_file" != *.fq.gz ]]; then
    echo "WARNING: Non-standard extension for FASTQ file: $input_file"
fi

echo "PASS: Input validation succeeded for $input_file"
SCRIPT
chmod +x /tmp/validate_input.sh

# Test with a valid file
echo "@read1" > /tmp/test.fastq
/tmp/validate_input.sh /tmp/test.fastq

# Test with missing file
/tmp/validate_input.sh /tmp/nonexistent.fastq

# Test with no arguments
/tmp/validate_input.sh

---

## 4. Loops

Loops are the reason you write Bash scripts. They let you process hundreds of files
with the same commands.

### for loop -- iterate over a list

In [ ]:
%%bash
# Iterate over explicit list
echo "=== Explicit list ==="
for sample in sample_01 sample_02 sample_03; do
    echo "Processing: $sample"
done

# Iterate over files matching a pattern
echo ""
echo "=== Files in /tmp matching *.sh ==="
for script in /tmp/*.sh; do
    echo "Found script: $(basename "$script")"
done

# Iterate over a range of numbers
echo ""
echo "=== Number range ==="
for i in {1..5}; do
    echo "  Chromosome $i"
done

# C-style for loop (useful when you need the index)
echo ""
echo "=== C-style loop ==="
for ((i=0; i<3; i++)); do
    echo "  Index: $i"
done

### Bioinformatics example: Batch processing FASTQ files

In [ ]:
%%bash
cat > /tmp/batch_qc.sh << 'SCRIPT'
#!/bin/bash
set -euo pipefail

INPUT_DIR="${1:-.}"       # First argument, default to current dir
OUTPUT_DIR="${2:-qc_results}"
THREADS=4

mkdir -p "$OUTPUT_DIR"

echo "Running QC on all FASTQ files in: $INPUT_DIR"
echo "Output directory: $OUTPUT_DIR"
echo "---"

count=0
for fastq in "${INPUT_DIR}"/*.fastq.gz; do
    # Check if the glob actually matched any files
    if [[ ! -f "$fastq" ]]; then
        echo "No .fastq.gz files found in $INPUT_DIR"
        exit 1
    fi

    sample=$(basename "$fastq" .fastq.gz)
    echo "[$((++count))] Processing: $sample"

    # In a real pipeline, you would run:
    # fastqc -t $THREADS -o "$OUTPUT_DIR" "$fastq"
    echo "    fastqc -t $THREADS -o $OUTPUT_DIR $fastq"
done

echo "---"
echo "Processed $count files"
SCRIPT

# Create test files
mkdir -p /tmp/test_fastq
for i in 1 2 3 4 5; do
    echo "@read" | gzip > /tmp/test_fastq/sample_${i}.fastq.gz
done

bash /tmp/batch_qc.sh /tmp/test_fastq /tmp/qc_output

### while loop -- condition-based iteration

In [ ]:
%%bash
# Basic while loop
count=1
while [[ $count -le 5 ]]; do
    echo "Iteration $count"
    ((count++))
done

echo ""

# Read a file line by line -- extremely common pattern
cat > /tmp/sample_list.txt << 'EOF'
sample_001	control
sample_002	control
sample_003	treatment
sample_004	treatment
EOF

echo "=== Reading sample list ==="
while IFS=$'\t' read -r sample_id condition; do
    echo "Sample: $sample_id  |  Condition: $condition"
done < /tmp/sample_list.txt

### Loop control: break and continue

In [ ]:
%%bash
# continue -- skip the rest of the current iteration
# break    -- exit the loop entirely

echo "=== Processing files (skip empty, stop if error) ==="

# Create some test files
echo "data" > /tmp/good_file.txt
touch /tmp/empty_file.txt              # Empty file
echo "more data" > /tmp/another_good.txt

for f in /tmp/good_file.txt /tmp/empty_file.txt /tmp/another_good.txt; do
    # Skip empty files
    if [[ ! -s "$f" ]]; then
        echo "  SKIP (empty): $(basename $f)"
        continue
    fi

    echo "  Processing: $(basename $f)"
done

---

## 5. Case Statements

When you need to handle multiple conditions based on a single value, `case` is
cleaner than a long chain of if/elif.

In [ ]:
%%bash
cat > /tmp/process_file.sh << 'SCRIPT'
#!/bin/bash
# Route bioinformatics files to the appropriate processing tool

file=$1

if [[ -z "$file" ]]; then
    echo "Usage: $0 <filename>"
    exit 1
fi

case "$file" in
    *.fastq.gz | *.fq.gz)
        echo "FASTQ file -> Running FastQC"
        # fastqc "$file"
        ;;
    *.bam)
        echo "BAM file -> Generating index and stats"
        # samtools index "$file"
        # samtools flagstat "$file"
        ;;
    *.vcf | *.vcf.gz)
        echo "VCF file -> Running bcftools stats"
        # bcftools stats "$file"
        ;;
    *.fasta | *.fa)
        echo "FASTA file -> Counting sequences"
        # grep -c "^>" "$file"
        ;;
    *.bed)
        echo "BED file -> Sorting"
        # sort -k1,1 -k2,2n "$file"
        ;;
    *)
        echo "Unknown file type: $file"
        exit 1
        ;;
esac
SCRIPT
chmod +x /tmp/process_file.sh

/tmp/process_file.sh reads.fastq.gz
/tmp/process_file.sh aligned.bam
/tmp/process_file.sh variants.vcf
/tmp/process_file.sh genome.fa

---

## 6. Functions

Functions let you write a block of code once and call it many times. They make
scripts modular, readable, and easier to debug.

In [ ]:
%%bash
# Define a logging function
log() {
    local timestamp=$(date '+%Y-%m-%d %H:%M:%S')
    echo "[$timestamp] $1"
}

# Define a function that takes arguments
count_sequences() {
    local file=$1
    if [[ ! -f "$file" ]]; then
        echo "0"
        return 1
    fi
    grep -c "^>" "$file"
}

# A function that checks if a tool is installed
require_tool() {
    local tool=$1
    if ! command -v "$tool" &> /dev/null; then
        log "ERROR: $tool is not installed"
        return 1
    fi
    log "Found: $tool ($(which $tool))"
}

# Use the functions
log "Starting analysis"
require_tool bash
require_tool python3

# Create a test FASTA
printf '>seq1\nACGT\n>seq2\nGCTA\n>seq3\nTTTT\n' > /tmp/test.fasta
n=$(count_sequences /tmp/test.fasta)
log "Found $n sequences in test.fasta"

### Local variables

Use `local` inside functions to avoid polluting the global scope:

```bash
my_function() {
    local result="computed value"   # Only visible inside this function
    echo "$result"
}
```

Without `local`, a variable set inside a function is visible everywhere in the script.

---

## 7. Error Handling

In bioinformatics, a silent failure in the middle of a pipeline can waste days of
compute time and produce garbage results. Defensive scripting catches errors early.

In [ ]:
%%bash
# === The most important line in any Bash script ===
# set -euo pipefail
#
# -e (errexit):   Exit immediately if ANY command returns non-zero
# -u (nounset):   Treat unset variables as errors (catches typos!)
# -o pipefail:    If any command in a pipe fails, the whole pipe fails
#                 (without this, only the LAST command's exit code matters)

# Example: without pipefail, this "succeeds" even though cat fails
echo "Without pipefail:"
cat /nonexistent/file 2>/dev/null | wc -l
echo "Exit code: $?"

echo ""

# With pipefail, it correctly reports failure
echo "With pipefail:"
set -o pipefail
cat /nonexistent/file 2>/dev/null | wc -l
echo "Exit code: $?"
set +o pipefail   # Turn it back off for the notebook

In [ ]:
%%bash
# === Trap: run cleanup code on exit or error ===

cat > /tmp/robust_script.sh << 'SCRIPT'
#!/bin/bash
set -euo pipefail

# Create a temp directory for intermediate files
TMPDIR=$(mktemp -d)

# Cleanup function -- runs on EXIT (normal or error)
cleanup() {
    echo "Cleaning up temp directory: $TMPDIR"
    rm -rf "$TMPDIR"
}
trap cleanup EXIT

# Main script -- uses temp directory
echo "Working in: $TMPDIR"
echo "some data" > "$TMPDIR/intermediate.txt"
echo "Temp file created: $(ls $TMPDIR)"

# Even if the script fails here, cleanup still runs
echo "Script completed successfully"
SCRIPT
bash /tmp/robust_script.sh

---

## 8. Text Processing in Scripts

Scripts often need to extract information from structured files, transform data
formats, and generate reports.

In [ ]:
%%bash
# === String manipulation in Bash ===

filename="sample_001_R1.fastq.gz"

# Extract parts of a filename
echo "Full name:  $filename"
echo "basename:   $(basename $filename)"             # sample_001_R1.fastq.gz
echo "dirname:    $(dirname /data/raw/$filename)"    # /data/raw
echo "No extension: ${filename%.fastq.gz}"           # sample_001_R1
echo "No path+ext:  $(basename $filename .fastq.gz)" # sample_001_R1

# Derive paired-end filename from R1
r1="sample_001_R1.fastq.gz"
r2="${r1/_R1/_R2}"             # Pattern substitution: _R1 -> _R2
echo ""
echo "R1: $r1"
echo "R2: $r2"

# Extract sample name
sample=$(basename "$r1" _R1.fastq.gz)
echo "Sample: $sample"

In [ ]:
%%bash
# === Using awk, sed, cut inside scripts ===

# Create a sample gene counts file
cat > /tmp/gene_counts.tsv << 'EOF'
gene_id	sample_1	sample_2	sample_3
BRCA1	150	230	180
TP53	500	480	520
MYC	50	45	60
GAPDH	10000	9800	10200
ACTB	8500	8700	8300
EOF

echo "=== Gene counts file ==="
cat /tmp/gene_counts.tsv

echo ""
echo "=== Genes with average count > 1000 ==="
awk -F'\t' 'NR>1 { avg=($2+$3+$4)/3; if(avg>1000) print $1, "avg="int(avg) }' /tmp/gene_counts.tsv

echo ""
echo "=== Normalized counts (divide by GAPDH) ==="
awk -F'\t' 'NR==1{print; next} $1=="GAPDH"{gapdh=$2; next} {printf "%s\t%.4f\t%.4f\t%.4f\n", $1, $2/10000, $3/10000, $4/10000}' /tmp/gene_counts.tsv

---

## 9. Automating BLAST Searches

BLAST (Basic Local Alignment Search Tool) is one of the most commonly used tools
in bioinformatics. Automating batch BLAST searches is a perfect use case for scripts.

In [ ]:
%%bash
cat > /tmp/batch_blast.sh << 'SCRIPT'
#!/bin/bash
set -euo pipefail

# ============================================================
# Batch BLAST Search
# Runs BLASTN on each FASTA file in the input directory
# ============================================================

QUERY_DIR="${1:-queries}"
DB="${2:-/databases/nt}"
OUTPUT_DIR="${3:-blast_results}"
THREADS=8
EVALUE="1e-5"
MAX_TARGETS=10

log() { echo "[$(date '+%H:%M:%S')] $1"; }

mkdir -p "$OUTPUT_DIR"

log "Starting batch BLAST"
log "Database: $DB"
log "E-value threshold: $EVALUE"

for query in "${QUERY_DIR}"/*.fasta; do
    if [[ ! -f "$query" ]]; then
        log "No FASTA files found in $QUERY_DIR"
        exit 1
    fi

    name=$(basename "$query" .fasta)
    output="${OUTPUT_DIR}/${name}_blast.tsv"

    # Skip if output already exists (avoid re-running)
    if [[ -f "$output" ]]; then
        log "SKIP (already done): $name"
        continue
    fi

    log "BLASTing: $name"

    # The actual BLAST command (commented out since we do not have the database)
    # blastn \
    #     -query "$query" \
    #     -db "$DB" \
    #     -out "$output" \
    #     -outfmt "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore" \
    #     -evalue "$EVALUE" \
    #     -max_target_seqs "$MAX_TARGETS" \
    #     -num_threads "$THREADS"

    echo "  blastn -query $query -db $DB -out $output -evalue $EVALUE"
done

log "Batch BLAST complete"
SCRIPT

# Create test queries
mkdir -p /tmp/blast_queries
echo -e ">gene1\nACGTACGTACGT" > /tmp/blast_queries/gene1.fasta
echo -e ">gene2\nGCTAGCTAGCTA" > /tmp/blast_queries/gene2.fasta
echo -e ">gene3\nTTTTAAAACCCC" > /tmp/blast_queries/gene3.fasta

bash /tmp/batch_blast.sh /tmp/blast_queries /databases/nt /tmp/blast_out

---

## 10. Complete Pipeline Script

Here is a full RNA-Seq pipeline script that demonstrates all the concepts from
this module: variables, functions, loops, conditionals, error handling, and
text processing.

In [ ]:
%%bash
cat > /tmp/rnaseq_pipeline.sh << 'PIPELINE'
#!/bin/bash
# ============================================================
# RNA-Seq Analysis Pipeline
# Processes paired-end FASTQ files through:
#   1. Quality control (FastQC)
#   2. Adapter trimming (Trimmomatic)
#   3. Alignment (HISAT2)
#   4. Read counting (featureCounts)
#
# Usage: ./rnaseq_pipeline.sh <input_dir> [output_dir]
# ============================================================

set -euo pipefail

# === Configuration ===
THREADS=8
INPUT_DIR="${1:-data/raw}"
OUTPUT_DIR="${2:-results}"
GENOME_INDEX="/references/hisat2/GRCh38"
ANNOTATION="/references/gencode.v38.gtf"
ADAPTER="AGATCGGAAGAGC"
MIN_LENGTH=36

# === Functions ===

log() {
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] $1"
}

check_dependencies() {
    local missing=0
    for tool in fastqc trimmomatic hisat2 samtools featureCounts; do
        if ! command -v "$tool" &> /dev/null; then
            log "ERROR: $tool not found in PATH"
            missing=1
        fi
    done
    if [[ $missing -eq 1 ]]; then
        log "Install missing tools and try again"
        exit 1
    fi
    log "All dependencies found"
}

run_qc() {
    local file=$1
    local outdir="${OUTPUT_DIR}/01_qc"
    mkdir -p "$outdir"
    log "  QC: $(basename $file)"
    # fastqc -t "$THREADS" -o "$outdir" "$file"
}

run_trimming() {
    local r1=$1
    local r2=$2
    local outdir="${OUTPUT_DIR}/02_trimmed"
    local sample=$(basename "$r1" _R1.fastq.gz)
    mkdir -p "$outdir"
    log "  Trimming: $sample"
    # trimmomatic PE -threads "$THREADS" \
    #     "$r1" "$r2" \
    #     "${outdir}/${sample}_R1.fastq.gz" "${outdir}/${sample}_R1_unpaired.fastq.gz" \
    #     "${outdir}/${sample}_R2.fastq.gz" "${outdir}/${sample}_R2_unpaired.fastq.gz" \
    #     ILLUMINACLIP:TruSeq3-PE-2.fa:2:30:10 \
    #     LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:"$MIN_LENGTH"
}

run_alignment() {
    local r1=$1
    local r2=$2
    local outdir="${OUTPUT_DIR}/03_aligned"
    local sample=$(basename "$r1" _R1.fastq.gz)
    mkdir -p "$outdir"
    log "  Aligning: $sample"
    # hisat2 -p "$THREADS" -x "$GENOME_INDEX" \
    #     -1 "$r1" -2 "$r2" 2> "${outdir}/${sample}_align.log" | \
    #     samtools sort -@ "$THREADS" -o "${outdir}/${sample}.bam" -
    # samtools index "${outdir}/${sample}.bam"
}

# === Main ===

log "========================================"
log "RNA-Seq Pipeline"
log "========================================"
log "Input:   $INPUT_DIR"
log "Output:  $OUTPUT_DIR"
log "Threads: $THREADS"
log "----------------------------------------"

# check_dependencies

mkdir -p "$OUTPUT_DIR"
total=0
failed=0

for r1 in "${INPUT_DIR}"/*_R1.fastq.gz; do
    if [[ ! -f "$r1" ]]; then
        log "ERROR: No *_R1.fastq.gz files found in $INPUT_DIR"
        exit 1
    fi

    r2="${r1/_R1/_R2}"
    sample=$(basename "$r1" _R1.fastq.gz)
    ((total++))

    if [[ ! -f "$r2" ]]; then
        log "WARNING: Missing R2 for $sample, skipping"
        ((failed++))
        continue
    fi

    log "Processing: $sample ($total)"

    # Step 1: QC on raw reads
    run_qc "$r1"
    run_qc "$r2"

    # Step 2: Trim adapters
    run_trimming "$r1" "$r2"

    # Step 3: Align to genome
    trimmed_r1="${OUTPUT_DIR}/02_trimmed/${sample}_R1.fastq.gz"
    trimmed_r2="${OUTPUT_DIR}/02_trimmed/${sample}_R2.fastq.gz"
    # run_alignment "$trimmed_r1" "$trimmed_r2"

    log "  Done: $sample"
done

log "----------------------------------------"
log "Pipeline complete!"
log "  Total samples:  $total"
log "  Failed/skipped: $failed"
log "========================================"
PIPELINE

# Create test paired-end FASTQ files
mkdir -p /tmp/rnaseq_raw
for s in patient_01 patient_02 patient_03; do
    echo "@read" | gzip > /tmp/rnaseq_raw/${s}_R1.fastq.gz
    echo "@read" | gzip > /tmp/rnaseq_raw/${s}_R2.fastq.gz
done

bash /tmp/rnaseq_pipeline.sh /tmp/rnaseq_raw /tmp/rnaseq_results

---

## 11. Arrays (Bonus)

Bash arrays are useful when you need to store lists of items.

In [ ]:
%%bash
# Declare an array
samples=("control_1" "control_2" "treated_1" "treated_2" "treated_3")

# Access elements
echo "First sample: ${samples[0]}"
echo "All samples:  ${samples[@]}"
echo "Count:        ${#samples[@]}"

# Loop over array
echo ""
echo "=== Processing all samples ==="
for sample in "${samples[@]}"; do
    echo "  Processing: $sample"
done

# Add to array
samples+=("treated_4")
echo ""
echo "After adding: ${samples[@]} (${#samples[@]} total)"

---

## 12. Useful Patterns and Idioms

These patterns come up repeatedly in bioinformatics scripts.

In [ ]:
%%bash
# === Default values for arguments ===
input="${1:-/data/default_input}"      # Use $1 if provided, else default
threads="${2:-4}"                       # Use $2 if provided, else 4

echo "Input: $input"
echo "Threads: $threads"

In [ ]:
%%bash
# === Parallel processing with xargs ===
# Run FastQC on 4 files at a time in parallel
# find data/ -name "*.fastq.gz" | xargs -P 4 -I {} fastqc {} -o results/qc/

# === GNU Parallel (if installed) ===
# parallel fastqc {} -o results/qc/ ::: data/*.fastq.gz

echo "Parallel processing patterns (run with real data)"

In [ ]:
%%bash
# === Progress reporting ===

total=10
for ((i=1; i<=total; i++)); do
    # \r returns to beginning of line, overwriting previous output
    printf "\rProcessing: %d/%d (%.0f%%)" "$i" "$total" "$((i*100/total))"
    sleep 0.1   # Simulating work
done
echo ""  # Newline at the end
echo "Complete!"

In [ ]:
%%bash
# === Generating a summary report ===

cat > /tmp/generate_report.sh << 'SCRIPT'
#!/bin/bash
# Generate a simple QC summary report

report_file="/tmp/qc_summary.txt"

{
    echo "QC Summary Report"
    echo "Generated: $(date)"
    echo "========================================"
    echo ""

    echo "Sample Files Found:"
    for f in /tmp/test_fastq/*.fastq.gz; do
        size=$(du -h "$f" | cut -f1)
        echo "  $(basename $f): $size"
    done

    echo ""
    echo "Total files: $(ls /tmp/test_fastq/*.fastq.gz | wc -l)"
    echo "Disk usage:  $(du -sh /tmp/test_fastq | cut -f1)"
} > "$report_file"

echo "Report saved to: $report_file"
cat "$report_file"
SCRIPT
bash /tmp/generate_report.sh

---

## Quick Reference Card

```
VARIABLES                       CONDITIONALS
  var="value"                      if [[ -f file ]]; then
  echo "$var"                      if [[ $a -eq $b ]]; then
  result=$(command)                if [[ "$s" == "text" ]]; then
  input="${1:-default}"            if [[ -z "$var" ]]; then

LOOPS                           FUNCTIONS
  for f in *.fastq.gz; do         my_func() {
      echo "$f"                       local arg=$1
  done                                echo "$arg"
                                  }
  while read -r line; do          my_func "value"
      echo "$line"
  done < file.txt

ERROR HANDLING                  STRING OPERATIONS
  set -euo pipefail               ${var%.ext}          # Remove suffix
  trap cleanup EXIT               ${var#prefix}        # Remove prefix
  if ! cmd; then                  ${var/old/new}       # Substitute
      echo "failed"               basename "$path"     # Filename only
  fi                              dirname "$path"      # Directory only

SCRIPT ARGUMENTS
  $0  Script name        $@  All arguments
  $1  First argument     $#  Number of arguments
  $?  Last exit code     $$  Process ID
```

---

## Exercises

### Exercise 1: Sequence File Processor

Write a script that takes a FASTA file as an argument and reports:
1. The number of sequences
2. The names of all sequences (headers without `>`)
3. The total number of nucleotides
4. The average sequence length

Include input validation (check that the file exists and is not empty).

In [ ]:
%%bash
# YOUR SOLUTION: Create the script


### Exercise 2: Sample Sheet Generator

Write a script that scans a directory of paired-end FASTQ files (named as
`samplename_R1.fastq.gz` and `samplename_R2.fastq.gz`) and generates a
tab-delimited sample sheet with columns: `sample_id`, `R1_path`, `R2_path`.

The script should warn if any R1 file is missing its R2 pair.

In [ ]:
%%bash
# YOUR SOLUTION:


### Exercise 3: FASTQ Quality Check

Write a function called `fastq_summary` that takes a FASTQ file (or `.gz`) and
prints:
1. Number of reads
2. Average read length
3. Whether it appears to be single-end or paired-end (based on the header format)

Then write a loop that calls this function on all `.fastq.gz` files in a directory.

In [ ]:
%%bash
# YOUR SOLUTION:


### Exercise 4: VCF Annotation Reporter

Write a script that takes a VCF file and produces a summary including:
1. Total number of variants (non-header lines)
2. Number of variants per chromosome
3. Number of PASS vs non-PASS variants
4. Distribution of variant types (SNP vs indel, based on REF/ALT length)

In [ ]:
%%bash
# YOUR SOLUTION:


### Exercise 5: Pipeline Wrapper

Write a complete pipeline script that:
1. Accepts an input directory and output directory as arguments
2. Validates inputs (directory exists, has FASTQ files)
3. Creates output subdirectories (qc, trimmed, aligned)
4. Loops over all paired-end FASTQ files
5. For each pair, runs (simulated) QC, trimming, and alignment steps
6. Logs progress with timestamps
7. Generates a summary report at the end
8. Uses `set -euo pipefail` and a cleanup trap

In [ ]:
%%bash
# YOUR SOLUTION:


---

## Exercise Solutions

### Solution 1

In [ ]:
%%bash
cat > /tmp/fasta_report.sh << 'SCRIPT'
#!/bin/bash
set -euo pipefail

file="${1:-}"

if [[ -z "$file" ]]; then
    echo "Usage: $0 <fasta_file>"
    exit 1
fi

if [[ ! -f "$file" ]]; then
    echo "ERROR: File not found: $file"
    exit 1
fi

if [[ ! -s "$file" ]]; then
    echo "ERROR: File is empty: $file"
    exit 1
fi

num_seqs=$(grep -c "^>" "$file")
total_nt=$(grep -v "^>" "$file" | tr -d '\n' | wc -c | tr -d ' ')
avg_len=$((total_nt / num_seqs))

echo "=== FASTA Report: $(basename $file) ==="
echo "Number of sequences: $num_seqs"
echo "Sequence names:"
grep "^>" "$file" | sed 's/^>/  /'
echo "Total nucleotides:   $total_nt"
echo "Average length:      $avg_len"
SCRIPT
chmod +x /tmp/fasta_report.sh

# Test with the sample FASTA from the Linux module
cat > /tmp/test_seqs.fasta << 'EOF'
>BRCA1 Homo sapiens
ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAAT
GCTATGCAGAAAATCTTAGAGTGTCCCATCTGTCTGGAGTTGATCAAGG
>TP53 Mus musculus
ATGACTGCCATGGAGGAGTCACAGTCGGATATCAGCCTCGAGCTCCCTC
>MYC Homo sapiens
ATGCCCCTCAACGTTAGCTTCACCAACAGGAACTATGACCTCGACTACGACTCGGTGCAG
CCGTATTTCTACTGCGACGAGGAGGAGAACTTCTACCAGCAGCAGCAGCAGAGCGAGCTG
EOF

/tmp/fasta_report.sh /tmp/test_seqs.fasta

### Solution 2

In [ ]:
%%bash
cat > /tmp/make_sample_sheet.sh << 'SCRIPT'
#!/bin/bash
set -euo pipefail

input_dir="${1:-.}"
output_file="${2:-sample_sheet.tsv}"

echo -e "sample_id\tR1_path\tR2_path" > "$output_file"

for r1 in "${input_dir}"/*_R1.fastq.gz; do
    if [[ ! -f "$r1" ]]; then
        echo "No *_R1.fastq.gz files found in $input_dir"
        exit 1
    fi

    r2="${r1/_R1/_R2}"
    sample=$(basename "$r1" _R1.fastq.gz)

    if [[ ! -f "$r2" ]]; then
        echo "WARNING: Missing R2 for $sample (expected: $r2)"
        continue
    fi

    echo -e "${sample}\t${r1}\t${r2}" >> "$output_file"
done

echo "Sample sheet written to: $output_file"
echo "---"
cat "$output_file"
SCRIPT

bash /tmp/make_sample_sheet.sh /tmp/rnaseq_raw /tmp/sample_sheet.tsv

### Solution 3

In [ ]:
%%bash
# Create test FASTQ files with realistic content
mkdir -p /tmp/fastq_test
for name in sample_A sample_B sample_C; do
    printf '@%s.1 length=50\n%s\n+\n%s\n@%s.2 length=50\n%s\n+\n%s\n' \
        "$name" "ACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTAC" \
        "IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII" \
        "$name" "GCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAG" \
        "IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII" | \
        gzip > "/tmp/fastq_test/${name}.fastq.gz"
done

fastq_summary() {
    local file=$1
    local num_reads=$(zcat "$file" | awk 'NR%4==1' | wc -l | tr -d ' ')
    local avg_len=$(zcat "$file" | awk 'NR%4==2 {sum+=length($0); n++} END {if(n>0) print int(sum/n); else print 0}')
    local first_header=$(zcat "$file" | head -1)

    # Check if paired based on header format (e.g., contains /1 or /2)
    local read_type="unknown"
    if [[ "$first_header" == *"/1"* || "$first_header" == *"/2"* ]]; then
        read_type="paired-end"
    else
        read_type="single-end (or interleaved)"
    fi

    echo "  File:       $(basename $file)"
    echo "  Reads:      $num_reads"
    echo "  Avg length: $avg_len bp"
    echo "  Type:       $read_type"
    echo ""
}

echo "=== FASTQ Summary ==="
for fq in /tmp/fastq_test/*.fastq.gz; do
    fastq_summary "$fq"
done

### Solution 4

In [ ]:
%%bash
cat > /tmp/sample_variants.vcf << 'EOF'
##fileformat=VCFv4.2
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO
chr1	100	.	A	G	50	PASS	.
chr1	200	.	C	T	30	PASS	.
chr1	300	.	ATG	A	25	LowQual	.
chr2	400	.	G	A	60	PASS	.
chr2	500	.	T	TCGA	15	LowQual	.
chr3	600	.	C	G	80	PASS	.
chr3	700	.	A	T	40	PASS	.
chrX	800	.	G	C	90	PASS	.
EOF

cat > /tmp/vcf_report.sh << 'SCRIPT'
#!/bin/bash
set -euo pipefail

vcf="${1:-}"
if [[ -z "$vcf" || ! -f "$vcf" ]]; then
    echo "Usage: $0 <vcf_file>"
    exit 1
fi

# Remove header lines for data processing
data=$(grep -v "^#" "$vcf")

total=$(echo "$data" | wc -l | tr -d ' ')

echo "=== VCF Summary: $(basename $vcf) ==="
echo "Total variants: $total"

echo ""
echo "Variants per chromosome:"
echo "$data" | cut -f1 | sort | uniq -c | sort -rn | awk '{printf "  %-8s %d\n", $2, $1}'

echo ""
echo "Filter status:"
echo "$data" | cut -f7 | sort | uniq -c | sort -rn | awk '{printf "  %-10s %d\n", $2, $1}'

echo ""
echo "Variant types:"
snps=$(echo "$data" | awk -F'\t' 'length($4)==1 && length($5)==1' | wc -l | tr -d ' ')
indels=$((total - snps))
echo "  SNPs:   $snps"
echo "  Indels: $indels"
SCRIPT

bash /tmp/vcf_report.sh /tmp/sample_variants.vcf

### Solution 5

In [ ]:
%%bash
cat > /tmp/full_pipeline.sh << 'SCRIPT'
#!/bin/bash
set -euo pipefail

# === Arguments ===
INPUT_DIR="${1:-}"
OUTPUT_DIR="${2:-results}"

# === Configuration ===
THREADS=4
LOGFILE="${OUTPUT_DIR}/pipeline.log"

# === Functions ===
log() {
    local msg="[$(date '+%Y-%m-%d %H:%M:%S')] $1"
    echo "$msg"
    echo "$msg" >> "$LOGFILE" 2>/dev/null || true
}

cleanup() {
    log "Cleaning up temporary files"
    rm -f /tmp/pipeline_*.tmp 2>/dev/null || true
}
trap cleanup EXIT

# === Input validation ===
if [[ -z "$INPUT_DIR" ]]; then
    echo "Usage: $0 <input_dir> [output_dir]"
    exit 1
fi

if [[ ! -d "$INPUT_DIR" ]]; then
    echo "ERROR: Input directory not found: $INPUT_DIR"
    exit 1
fi

# Check for FASTQ files
n_files=$(ls "${INPUT_DIR}"/*_R1.fastq.gz 2>/dev/null | wc -l)
if [[ $n_files -eq 0 ]]; then
    echo "ERROR: No *_R1.fastq.gz files found in $INPUT_DIR"
    exit 1
fi

# === Setup ===
mkdir -p "${OUTPUT_DIR}"/{qc,trimmed,aligned}
touch "$LOGFILE"

log "=============================="
log "Pipeline started"
log "Input:   $INPUT_DIR"
log "Output:  $OUTPUT_DIR"
log "Samples: $n_files"
log "=============================="

# === Process each sample ===
processed=0
skipped=0
start_time=$(date +%s)

for r1 in "${INPUT_DIR}"/*_R1.fastq.gz; do
    r2="${r1/_R1/_R2}"
    sample=$(basename "$r1" _R1.fastq.gz)

    if [[ ! -f "$r2" ]]; then
        log "WARNING: Skipping $sample (missing R2)"
        ((skipped++))
        continue
    fi

    log "Processing: $sample"

    # Step 1: QC
    log "  Step 1: Quality control"
    # fastqc -t $THREADS -o "${OUTPUT_DIR}/qc" "$r1" "$r2"

    # Step 2: Trimming
    log "  Step 2: Adapter trimming"
    # trimmomatic PE -threads $THREADS "$r1" "$r2" ...

    # Step 3: Alignment
    log "  Step 3: Alignment"
    # hisat2 -p $THREADS -x genome -1 r1_trimmed -2 r2_trimmed | samtools sort ...

    log "  Done: $sample"
    ((processed++))
done

end_time=$(date +%s)
elapsed=$((end_time - start_time))

# === Summary report ===
report="${OUTPUT_DIR}/summary_report.txt"
{
    echo "Pipeline Summary Report"
    echo "======================="
    echo "Date:      $(date)"
    echo "Input:     $INPUT_DIR"
    echo "Output:    $OUTPUT_DIR"
    echo "Processed: $processed"
    echo "Skipped:   $skipped"
    echo "Runtime:   ${elapsed}s"
} > "$report"

log "=============================="
log "Pipeline complete!"
log "Processed: $processed samples"
log "Skipped:   $skipped samples"
log "Runtime:   ${elapsed}s"
log "Report:    $report"
log "=============================="
SCRIPT

bash /tmp/full_pipeline.sh /tmp/rnaseq_raw /tmp/pipeline_output
echo ""
echo "=== Summary Report ==="
cat /tmp/pipeline_output/summary_report.txt

---

## Key Takeaways

1. **Start with `set -euo pipefail`**: This single line prevents most silent failures
2. **Always quote variables**: `"$var"` prevents word-splitting bugs
3. **Use functions**: Keep your scripts modular and readable
4. **Validate inputs**: Check that files exist, arguments are provided, and tools are installed
5. **Log everything**: Add timestamps to output, especially for long-running jobs
6. **Use variables for configuration**: Put paths, thresholds, and parameters at the top
7. **Test on small data first**: Run on 2-3 samples before processing 200

---

### Next: Tier 1 - Python Fundamentals -->

---

[← Previous: Module 0.2: Git Version Control for Scientists](../../Tier_0_Computational_Foundations/02_Git_Version_Control/01_git_version_control.ipynb) | [Next: Character Encodings and Binary Data in Bioinformatics →](../../Tier_0_Computational_Foundations/04_File_Encodings/01_character_encodings.ipynb)